In [1]:
import pandas as pd

df = pd.read_csv("../data/clean_products.csv")

print(df.shape)


(1666, 24)


In [2]:
df.columns.tolist()

['id',
 'slug',
 'title',
 'imgs',
 'brand',
 'category',
 'vendor',
 'used',
 'address',
 'availability',
 'currency',
 'original_price',
 'discounted_price',
 'specifications',
 'description',
 'delivery_fee',
 'delivery_details',
 'warranty',
 'warranty_type',
 'average_rating',
 'num_ratings',
 'reviews',
 'search_text',
 'clean_search_text']

In [3]:
brands = (
    df["brand"]
    .dropna()
    .unique()
    .tolist()
)

print(brands[:20])
print(len(brands))

['OPPO', 'Tecno', 'Vivo', 'Apple', 'Realme', 'Sparx', 'OnePlus', 'Samsung', 'Infinix', 'Google', 'Red Magic', 'Motorola', 'Nokia', 'Xiaomi', 'Acer', 'Dell', 'Lenovo', 'Asus', 'HP', 'Microsoft']
22


In [4]:
categories = (
    df["category"]
    .dropna()
    .unique()
    .tolist()
)

print(categories)

['Mobile', 'Laptop', 'Earbuds', 'Watch']


In [5]:
def extract_brand(query):

    query = query.lower()

    for brand in brands:

        if str(brand).lower() in query:
            return brand

    return None

In [12]:
def extract_category(query):

    query = query.lower()

    for category, synonyms in CATEGORY_SYNONYMS.items():

        for synonym in synonyms:

            if synonym in query:
                return category

    return None

In [7]:
def detect_intent(query):

    query = query.lower()

    if any(
        word in query
        for word in [
            "cheap",
            "cheapest",
            "lowest price"
        ]
    ):
        return "cheapest"

    if any(
        word in query
        for word in [
            "budget",
            "affordable"
        ]
    ):
        return "budget"

    if any(
        word in query
        for word in [
            "best",
            "top",
            "highest rated"
        ]
    ):
        return "quality"

    return "general"

In [8]:
def parse_query(query):

    return {
        "brand": extract_brand(query),
        "category": extract_category(query),
        "intent": detect_intent(query)
    }

In [9]:
parse_query("budget samsung mobile")

{'brand': 'Samsung', 'category': 'Mobile', 'intent': 'budget'}

In [10]:
parse_query("best apple phone")

{'brand': 'Apple', 'category': None, 'intent': 'quality'}

In [11]:
CATEGORY_SYNONYMS = {
    "Mobile": [
        "mobile",
        "phone",
        "smartphone",
        "cell phone"
    ],

    "Laptop": [
        "laptop",
        "notebook"
    ],

    "Earbuds": [
        "earbuds",
        "earphones",
        "headphones",
        "buds"
    ],

    "Watch": [
        "watch",
        "smartwatch"
    ]
}

In [13]:
parse_query("best apple phone")

{'brand': 'Apple', 'category': 'Mobile', 'intent': 'quality'}

In [14]:
parse_query("wireless earbuds")

{'brand': None, 'category': 'Earbuds', 'intent': 'general'}

In [15]:
parse_query(
    "budget samsung mobile"
)

{'brand': 'Samsung', 'category': 'Mobile', 'intent': 'budget'}

In [17]:
parsed = parse_query("budget samsung mobile")

parsed

{'brand': 'Samsung', 'category': 'Mobile', 'intent': 'budget'}

In [18]:
query = "budget samsung mobile"

parsed = parse_query(query)

parsed

{'brand': 'Samsung', 'category': 'Mobile', 'intent': 'budget'}

In [19]:
working_df = df.copy()

if parsed["brand"]:
    working_df = working_df[
        working_df["title"]
        .str.contains(
            parsed["brand"],
            case=False,
            na=False
        )
    ]

if parsed["category"]:
    working_df = working_df[
        working_df["category"]
        .str.lower()
        ==
        parsed["category"].lower()
    ]

print(working_df.shape)

(64, 24)


In [20]:
parse_query("budget samsung mobile")

{'brand': 'Samsung', 'category': 'Mobile', 'intent': 'budget'}

In [21]:
print(working_df.shape)

(64, 24)


In [22]:
working_df[
    [
        "title",
        "brand",
        "category"
    ]
].head(10)

,title,brand,category
12,Samsung Galaxy S23 Ultra 12GB RAM 512GB Storag...,Samsung,Mobile
18,Samsung Galaxy Z Fold 4 12GB Ram 512GB Storage...,Samsung,Mobile
22,Samsung Galaxy A14 6GB RAM 128GB Storage PTA A...,Samsung,Mobile
23,Samsung Galaxy A14 4GB RAM 128GB Storage PTA A...,Samsung,Mobile
24,Samsung Galaxy Note 20 Ultra 12GB RAM 128GB St...,Samsung,Mobile
34,Samsung Galaxy S23 Plus 8GB RAM 256GB Storage ...,Samsung,Mobile
35,Samsung Galaxy S23 8GB RAM 256GB Storage Non PTA,Samsung,Mobile
37,Samsung Galaxy S23 Ultra 8GB RAM 256GB Storage...,Samsung,Mobile
43,Samsung Galaxy A04 4GB RAM 64GB Storage PTA Ap...,Samsung,Mobile
44,Samsung Galaxy A04 3GB RAM 32GB Storage PTA Ap...,Samsung,Mobile


In [24]:
import sys

sys.path.append("..")

In [25]:
from src.query_parser import (
    get_brands,
    parse_query
)

In [26]:
import os

print(os.getcwd())

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks


In [27]:
sys.path.append("..")

In [28]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [29]:
from src.query_parser import (
    get_brands,
    parse_query
)

In [30]:
from src.query_parser import (
    get_brands,
    parse_query
)

brands = get_brands(df)

print(
    parse_query(
        "budget samsung mobile",
        brands
    )
)

print(
    parse_query(
        "best apple phone",
        brands
    )
)

print(
    parse_query(
        "wireless earbuds",
        brands
    )
)

{'brand': 'Samsung', 'category': 'Mobile', 'intent': 'budget'}
{'brand': 'Apple', 'category': 'Mobile', 'intent': 'quality'}
{'brand': None, 'category': 'Earbuds', 'intent': 'general'}


In [31]:
from src.search_engine import search_products_v3

In [33]:
import sys
sys.path.append("..")

import pandas as pd
import pickle

from src.preprocessing import clean_text

In [39]:
import pickle

with open("../models/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("../models/tfidf_matrix.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)

print(type(vectorizer))
print(tfidf_matrix.shape)

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
(1666, 4513)


In [40]:
import pandas as pd

df = pd.read_csv("../data/clean_products.csv")

print(df.shape)

(1666, 24)


In [36]:
import os

print(os.getcwd())

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks


In [37]:
import os

print(os.listdir("../models"))

['products.pkl', 'product_embeddings.pkl', 'README.md', 'tfidf_matrix.pkl', 'vectorizer.pkl']


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

df["clean_search_text"] = (
    df["search_text"]
    .fillna("")
    .apply(clean_text)
)

vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(
    df["clean_search_text"]
)

print(tfidf_matrix.shape)

(1666, 4513)


In [41]:
import sys

sys.path.append("..")

from src.preprocessing import clean_text


In [43]:
import numpy as np

df["rating_score"] = (
    df["average_rating"].fillna(0) / 5
)

df["popularity_score"] = (
    np.log1p(
        df["num_ratings"].fillna(0)
    )
)

# Normalize popularity

df["popularity_score"] = (
    df["popularity_score"]
    /
    df["popularity_score"].max()
)

In [44]:
df[
    [
        "average_rating",
        "rating_score",
        "num_ratings",
        "popularity_score"
    ]
].head()

,average_rating,rating_score,num_ratings,popularity_score
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0


In [45]:
results = search_products_v3(
    "budget samsung mobile",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

results[:5]

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks\..\src\search_engine.py:179: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output = output.replace(


[{'title': 'Samsung Galaxy A12',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 319.0,
  'similarity_score': 0.13790030014790877,
  'final_score': 0.22023989713457542},
 {'title': 'Samsung Galaxy A03',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 177.0,
  'similarity_score': 0.13939161469638167,
  'final_score': 0.21892515376996083},
 {'title': 'Samsung Galaxy A13',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 229.0,
  'similarity_score': 0.13764206822938582,
  'final_score': 0.21851154320919744},
 {'title': 'Samsung Galaxy A22',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 73.0,
  'similarity_score': 0.13966248992592045,
  'final_score': 0.21519300880693085},
 {'title': 'Samsung Galaxy A04',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 60.0,
  'similarity_score': 0.1405951083142395,
  'final_score': 0.21515723

In [46]:
import pickle

with open("../models/product_embeddings.pkl", "rb") as f:
    product_embeddings = pickle.load(f)

print(type(product_embeddings))
print(product_embeddings.shape)

<class 'numpy.ndarray'>
(1666, 384)


In [49]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded")

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1393.91it/s]


Model Loaded


In [50]:
query = "best apple phone"

query_embedding = model.encode(
    [query]
)

print(query_embedding.shape)

(1, 384)


In [51]:
parsed = parse_query(
    "budget samsung mobile",
    brands
)

working_df = df.copy()

if parsed["brand"]:
    working_df = working_df[
        working_df["title"]
        .str.contains(
            parsed["brand"],
            case=False,
            na=False
        )
    ]

if parsed["category"]:
    working_df = working_df[
        working_df["category"]
        .str.lower()
        ==
        parsed["category"].lower()
    ]

candidate_indices = working_df.index.tolist()

print(len(candidate_indices))

64


In [52]:
parsed = parse_query(
    "budget samsung mobile",
    brands
)

working_df = df.copy()

if parsed["brand"]:
    working_df = working_df[
        working_df["title"]
        .str.contains(
            parsed["brand"],
            case=False,
            na=False
        )
    ]

if parsed["category"]:
    working_df = working_df[
        working_df["category"]
        .str.lower()
        ==
        parsed["category"].lower()
    ]

candidate_indices = working_df.index.tolist()

print(len(candidate_indices))

64


In [53]:
candidate_embeddings = product_embeddings[
    candidate_indices
]

print(candidate_embeddings.shape)

(64, 384)


In [54]:
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
query = "budget samsung mobile"

query_embedding = model.encode(
    [query]
)

In [56]:
semantic_scores = cosine_similarity(
    query_embedding,
    candidate_embeddings
).flatten()

print(semantic_scores.shape)

(64,)


In [57]:
semantic_df = working_df.copy()

semantic_df["semantic_score"] = semantic_scores

In [58]:
semantic_df.sort_values(
    "semantic_score",
    ascending=False
).head(10)

,id,slug,title,imgs,brand,category,vendor,used,address,availability,...,warranty,warranty_type,average_rating,num_ratings,reviews,search_text,clean_search_text,rating_score,popularity_score,semantic_score
1571,1571,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A32,['https://images.priceoye.pk/samsung-galaxy-a3...,NaN,Mobile,PriceOye,0,NaN,Avaiable,...,NaN,NaN,5.0,751.0,"[""it was my first order that I placed from pri...",Samsung Galaxy A32 Mobile {'Release Date': '2...,samsung galaxy a32 mobile release date 2021 02...,1.0,1.000000,0.613588
1351,1351,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A33 5G,['https://images.priceoye.pk/samsung-galaxy-a3...,NaN,Mobile,PriceOye,0,NaN,Avaiable,...,NaN,NaN,5.0,98.0,['Nice genuine sealed product with official wa...,Samsung Galaxy A33 5G Mobile {'Release Date':...,samsung galaxy a33 5g mobile release date 2021...,1.0,0.693840,0.608919
1576,1576,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A22,['https://images.priceoye.pk/samsung-galaxy-a2...,NaN,Mobile,PriceOye,0,NaN,Not Available,...,NaN,NaN,5.0,73.0,['Good service. Right original product with al...,Samsung Galaxy A22 Mobile {'Release Date': '2...,samsung galaxy a22 mobile release date 2021 03...,1.0,0.649892,0.604782
1582,1582,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A12,['https://images.priceoye.pk/samsung-galaxy-a1...,NaN,Mobile,PriceOye,0,NaN,Not Available,...,NaN,NaN,5.0,319.0,['Awesome yaar kya aap hai yaar what happened ...,Samsung Galaxy A12 Mobile {'Release Date': '2...,samsung galaxy a12 mobile release date 2020 11...,1.0,0.870988,0.601893
1344,1344,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A13,['https://images.priceoye.pk/samsung-galaxy-a1...,NaN,Mobile,PriceOye,0,NaN,Avaiable,...,NaN,NaN,5.0,229.0,['Alhamdulillah received Really nice device De...,Samsung Galaxy A13 Mobile {'Release Date': '2...,samsung galaxy a13 mobile release date 2022 04...,1.0,0.821123,0.599219
1353,1353,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A53 5G,['https://images.priceoye.pk/samsung-galaxy-a5...,NaN,Mobile,PriceOye,0,NaN,Avaiable,...,NaN,NaN,5.0,91.0,['It does tell your savings on the order but ...,Samsung Galaxy A53 5G Mobile {'Release Date':...,samsung galaxy a53 5g mobile release date 2021...,1.0,0.682767,0.594230
1587,1587,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A03,['https://images.priceoye.pk/samsung-galaxy-a0...,NaN,Mobile,PriceOye,0,NaN,Avaiable,...,NaN,NaN,5.0,177.0,['Excellent packing and sms system was super! ...,Samsung Galaxy A03 Mobile {'Release Date': '2...,samsung galaxy a03 mobile release date 2022 04...,1.0,0.782423,0.594018
1429,1429,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy S22,['https://images.priceoye.pk/samsung-galaxy-s2...,NaN,Mobile,PriceOye,0,NaN,Avaiable,...,NaN,NaN,5.0,18.0,['Delivery was late but mobile. Its packing an...,Samsung Galaxy S22 Mobile {'Release Date': '2...,samsung galaxy s22 mobile release date 2022 05...,1.0,0.444596,0.593337
494,494,https://www.czone.com.pk/tablet-pc-samsung-tab...,"Samsung Galaxy Tab A8 SM-X205 10.5"" Tablet 64G...",['https://www.czone.com.pk/images/thumbnails-l...,Samsung,Mobile,ComputerZone,0,"FL 4/20, Main Rashid Minhas Road, Gulshan-e-Iq...",Available on Order,...,NaN,NaN,0.0,0.0,[],"Samsung Galaxy Tab A8 SM-X205 10.5"" Tablet 64G...",samsung galaxy tab a8 sm x205 10 5 tablet 64gb...,0.0,0.000000,0.592493
1388,1388,https://priceoye.pk/mobiles/samsung/samsung-ga...,Samsung Galaxy A23,['https://images.priceoye.pk/samsung-galaxy-a2...,NaN,Mobile,PriceOye,0,NaN,Avaiable,...,NaN,NaN,5.0,91.0,['Excellent Service with best discount price f...,Samsung Galaxy A23 Mobile {'Release Date': '2...,samsung galaxy a23 mobile release date 2022 04...,1.0,0.682767,0.587707


In [59]:
semantic_df.sort_values(
    "semantic_score",
    ascending=False
)[
    [
        "title",
        "semantic_score"
    ]
].head(10)

,title,semantic_score
1571,Samsung Galaxy A32,0.613588
1351,Samsung Galaxy A33 5G,0.608919
1576,Samsung Galaxy A22,0.604782
1582,Samsung Galaxy A12,0.601893
1344,Samsung Galaxy A13,0.599219
1353,Samsung Galaxy A53 5G,0.594230
1587,Samsung Galaxy A03,0.594018
1429,Samsung Galaxy S22,0.593337
494,"Samsung Galaxy Tab A8 SM-X205 10.5"" Tablet 64G...",0.592493
1388,Samsung Galaxy A23,0.587707


In [60]:
search_products_v3(
    "best apple phone",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)[:5]

[{'title': 'Apple iPhone 13',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 6.0,
  'similarity_score': 0.1892519472311442,
  'final_score': 0.24914143296881594},
 {'title': 'Apple iPhone 11',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 127.0,
  'similarity_score': 0.1668264176710004,
  'final_score': 0.24212274362924913},
 {'title': 'Apple iPad Pro 11" - Apple M2 Chip',
  'brand': 'Apple',
  'category': 'Mobile',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.2505364225146494,
  'final_score': 0.22548278026318447},
 {'title': 'Apple iPhone 14 Pro Max',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 6.0,
  'similarity_score': 0.14598715713873867,
  'final_score': 0.21020312188565096},
 {'title': 'Apple iPhone 14 Pro',
  'brand': '',
  'category': 'Mobile',
  'average_rating': 5.0,
  'num_ratings': 3.0,
  'similarity_score': 0.14327356641093825,
  'final_score